In [5]:
import os

# Force Python path
os.environ["PYSPARK_PYTHON"] = r"C:\Users\Nishanth\anaconda3\envs\pyspark_env\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\Nishanth\anaconda3\envs\pyspark_env\python.exe"

# CRITICAL FIX (network binding)
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.driver.host=127.0.0.1 --conf spark.driver.bindAddress=127.0.0.1 pyspark-shell"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("RDD Impl") \
    .master("local[2]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.python.worker.reuse", "false") \ 
    .getOrCreate()

# Create RDD
rdd = sc.parallelize([1, 2, 3, 4, 5])

print(rdd.collect())

[1, 2, 3, 4, 5]


In [9]:
spark.stop()

In [1]:
import os

os.environ["PYSPARK_PYTHON"] = r"C:\Users\Nishanth\anaconda3\envs\pyspark_env\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\Nishanth\anaconda3\envs\pyspark_env\python.exe"

# Stronger network fix
os.environ["PYSPARK_SUBMIT_ARGS"] = """
--conf spark.driver.host=127.0.0.1 
--conf spark.driver.bindAddress=127.0.0.1 
--conf spark.python.worker.reuse=false 
--conf spark.network.timeout=600s 
--conf spark.executor.heartbeatInterval=60s 
pyspark-shell
"""

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("StableRDD") \
    .master("local[2]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.network.timeout", "600s") \
    .getOrCreate()

sc = spark.sparkContext

In [3]:
#Basic RDD Transformations
rdd = sc.parallelize([1, 2, 3, 4])
squared = rdd.map(lambda x: x * x)
print(squared.collect())

[1, 4, 9, 16]


In [4]:
#filter
even = rdd.filter(lambda x: x % 2 == 0)

print(even.collect())

[2, 4]


In [5]:
#FlatMap
data = ["hello world", "spark is powerful"]

rdd = sc.parallelize(data)

words = rdd.flatMap(lambda x: x.split(" "))

print(words.collect())

['hello', 'world', 'spark', 'is', 'powerful']


In [7]:
#Key-Value RDD Operations
data = [("a", 1), ("b", 2), ("a", 3), ("b", 4)]

rdd = sc.parallelize(data)

In [8]:
#reduceByKey()
result = rdd.reduceByKey(lambda x, y: x + y)

print(result.collect())

[('b', 6), ('a', 4)]


In [9]:
#groupByKey() (costly)
result = rdd.groupByKey().mapValues(list)

print(result.collect())

[('b', [2, 4]), ('a', [1, 3])]


In [10]:
rdd = sc.parallelize([10, 20, 30])

print(rdd.count())
print(rdd.first())
print(rdd.take(2))
print(rdd.reduce(lambda x, y: x + y))

3
10
[10, 20]
60


In [ ]:
#counting the number of words and max words
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("StableRDD") \
    .master("local[2]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.network.timeout", "600s") \
    .getOrCreate()

sc = spark.sparkContext

# Load text file
rdd = sc.textFile("word_count_text.txt")

# Split into words
words = rdd.flatMap(lambda line: line.lower().split())

# Remove punctuation (basic cleaning)
import re
words_clean = words.map(lambda word: re.sub(r'[^a-z]', '', word)) \
                   .filter(lambda word: word != "")

# Word Count
word_count = words_clean.map(lambda word: (word, 1)) \
                        .reduceByKey(lambda a, b: a + b)

print("Word Count")
word_count.collect()

# Max Repeated Word
max_word = word_count.reduce(lambda a, b: a if a[1] > b[1] else b)

print("=== Most Repeated Word ===")
print(max_word)

TRY
#Find longest word
#Find average word length

In [11]:
#Top 3 Products by Total Sales
rdd = sc.textFile("sales_data.csv")

# Remove header
header = rdd.first()
data = rdd.filter(lambda x: x != header)

# Parse → (product, amount)
product_sales = data.map(lambda x: x.split(",")) \
                    .map(lambda x: (x[1], int(x[2])))

# Aggregate
total_sales = product_sales.reduceByKey(lambda x, y: x + y)

# Sort descending and take top 3
top3 = total_sales.sortBy(lambda x: x[1], ascending=False).take(3)

print(top3)

[('D', 36707), ('A', 36605), ('C', 36016)]


In [12]:
#Second Highest Sale Per Country
rdd = sc.textFile("sales_data.csv")

header = rdd.first()
data = rdd.filter(lambda x: x != header)

parsed = data.map(lambda x: x.split(",")) \
             .map(lambda x: (x[0], int(x[2])))

# Group values
grouped = parsed.groupByKey()

# Find second highest
second_highest = grouped.mapValues(lambda vals: sorted(vals, reverse=True)[1] if len(vals) > 1 else None)

print(second_highest.collect())

[('USA', 490), ('India', 492), ('UK', 487), ('Germany', 495)]


In [13]:
#Remove Duplicates WITHOUT distinct()
rdd = sc.parallelize([1,2,2,3,4,4,5])

unique = rdd.map(lambda x: (x, 1)) \
            .reduceByKey(lambda x, y: x) \
            .map(lambda x: x[0])

print(unique.collect())

[2, 4, 1, 3, 5]


In [14]:
#Total Spending Per User
import json

rdd = sc.textFile("nested_data.json")

data = rdd.map(lambda x: json.loads(x))

# Flatten
user_spend = data.flatMap(lambda user: [
    (user["user"], item["price"])
    for order in user["orders"]
    for item in order["items"]
])

# Aggregate
total = user_spend.reduceByKey(lambda x, y: x + y)

print(total.collect())

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 1 in stage 28.0 failed 1 times, most recent failure: Lost task 1.0 in stage 28.0 (TID 42) (DESKTOP-Q6MNOT9 executor driver): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "C:\spark\spark-4.1.1-bin-hadoop3\spark-4.1.1-bin-hadoop3\python\lib\pyspark.zip\pyspark\worker.py", line 3386, in main
  File "C:\spark\spark-4.1.1-bin-hadoop3\spark-4.1.1-bin-hadoop3\python\lib\pyspark.zip\pyspark\worker.py", line 3375, in process
  File "C:\Users\Nishanth\anaconda3\envs\pyspark_env\lib\site-packages\pyspark\core\rdd.py", line 5306, in pipeline_func
    return func(split, prev_func(split, iterator))
  File "C:\Users\Nishanth\anaconda3\envs\pyspark_env\lib\site-packages\pyspark\core\rdd.py", line 5306, in pipeline_func
    return func(split, prev_func(split, iterator))
  File "C:\Users

In [15]:
##Total Spending Per User
import json

rdd = sc.wholeTextFiles("nested_data.json")

# Extract content
data = rdd.flatMap(lambda x: json.loads(x[1]))

# Now continue
user_spend = data.flatMap(lambda user: [
    (user["user"], item["price"])
    for order in user["orders"]
    for item in order["items"]
])

total = user_spend.reduceByKey(lambda x, y: x + y)

print(total.collect())

[('user_0', 137), ('user_1', 284), ('user_2', 69), ('user_3', 41), ('user_4', 304), ('user_5', 291), ('user_6', 278), ('user_7', 325), ('user_8', 54), ('user_9', 222), ('user_10', 296), ('user_11', 315), ('user_12', 105), ('user_13', 218), ('user_14', 39), ('user_15', 205), ('user_16', 365), ('user_17', 112), ('user_18', 250), ('user_19', 117), ('user_20', 241), ('user_21', 324), ('user_22', 16), ('user_23', 166), ('user_24', 40), ('user_25', 396), ('user_26', 109), ('user_27', 120), ('user_28', 205), ('user_29', 39), ('user_30', 269), ('user_31', 316), ('user_32', 147), ('user_33', 247), ('user_34', 209), ('user_35', 273), ('user_36', 123), ('user_37', 181), ('user_38', 92), ('user_39', 239), ('user_40', 337), ('user_41', 81), ('user_42', 319), ('user_43', 115), ('user_44', 18), ('user_45', 300), ('user_46', 189), ('user_47', 95), ('user_48', 98), ('user_49', 412)]


In [ ]:
Question 5: Most Sold Product (Global)

In [3]:
rdd = sc.textFile("sales_data.csv")

header = rdd.first()
data = rdd.filter(lambda x: x != header)

product_count = data.map(lambda x: x.split(",")[1]) \
                    .map(lambda x: (x, 1)) \
                    .reduceByKey(lambda x, y: x + y)

top_product = product_count.sortBy(lambda x: x[1], ascending=False).first()

print(top_product)

('C', 136)


In [ ]:
#Question : Join Two RDDs

In [4]:
users = sc.parallelize([
    ("u1", "Nishanth"),
    ("u2", "John")
])

transactions = sc.parallelize([
    ("u1", 100),
    ("u2", 200),
    ("u1", 50)
])

joined = users.join(transactions)

print(joined.collect())

[('u2', ('John', 200)), ('u1', ('Nishanth', 100)), ('u1', ('Nishanth', 50))]


In [ ]:
#Question : Detect Data Skew

In [5]:
#Detect Data Skew
rdd = sc.textFile("sales_data.csv")

header = rdd.first()
data = rdd.filter(lambda x: x != header)

country_count = data.map(lambda x: x.split(",")[0]) \
                    .map(lambda x: (x, 1)) \
                    .reduceByKey(lambda x, y: x + y)

# Detect skew (threshold example: > 100)
skewed = country_count.filter(lambda x: x[1] > 100)

print(skewed.collect())

[('USA', 130), ('India', 119), ('UK', 111), ('Germany', 140)]


In [ ]:
from pyspark.sql import SparkSession
import re

# Create Spark Session
spark = SparkSession.builder.appName("WordAnalysisRDD").getOrCreate()
sc = spark.sparkContext

# Load file
rdd = sc.textFile("word_count_text.txt")

# Split into words
words = rdd.flatMap(lambda line: line.lower().split())

# Clean words (remove punctuation)
words_clean = words.map(lambda w: re.sub(r'[^a-z]', '', w)) \
                   .filter(lambda w: w != "")


#  Longest Word

longest_word = words_clean.reduce(lambda a, b: a if len(a) > len(b) else b)

print("Longest Word:", longest_word)
print("Length:", len(longest_word))



# 2 Average Word Length 
# Convert each word to (length, 1)
length_rdd = words_clean.map(lambda w: (len(w), 1))

# Sum lengths and counts
total_length, total_words = length_rdd.reduce(lambda a, b: (a[0] + b[0], a[1] + b[1]))

average_length = total_length / total_words

print("Average Word Length:", average_length)